## Feature Engineering and Preliminary Feature Selection

Build a patient-level dataset from file-level features and run leakage-safe feature selection (train split only).

In [2]:
# Load features generated in 01_data_exploration (section 7.3)
processed_dir = Path('../..') / 'data' / 'processed'
parquet_path = processed_dir / 'timeseries_features.parquet'
csv_path = processed_dir / 'timeseries_features.csv'

if parquet_path.exists():
    df_features = pd.read_parquet(parquet_path)
elif csv_path.exists():
    df_features = pd.read_csv(csv_path)
else:
    raise FileNotFoundError('Feature table not found. Run section 7.3 in notebook 01 first.')

# Load patient demographics
base_path = Path('../..') / 'data' / 'raw' / 'pads-parkinsons-disease-smartwatch-dataset-1.0.0'
patients_path = base_path / 'patients'
questionnaire_path = base_path / 'questionnaire'

patient_files = sorted(list(patients_path.glob('*.json')))
questionnaire_files = sorted(list(questionnaire_path.glob('*.json')))

import json
patient_rows = []
for fp in patient_files:
    with open(fp, 'r') as f:
        patient_rows.append(json.load(f))
df_patients = pd.DataFrame(patient_rows)

# Load questionnaire items (subject-level yes/no responses)
questionnaire_rows = []
for fp in questionnaire_files:
    with open(fp, 'r') as f:
        q = json.load(f)
    subject_id = q.get('subject_id')
    questionnaire_name = q.get('questionnaire_name')
    for item in q.get('item', []):
        questionnaire_rows.append({
            'subject_id': subject_id,
            'questionnaire_name': questionnaire_name,
            'link_id': item.get('link_id'),
            'answer': item.get('answer')
        })

df_questionnaire_items = pd.DataFrame(questionnaire_rows)
if len(df_questionnaire_items) > 0:
    df_questionnaire_items['answer_num'] = df_questionnaire_items['answer'].map({True: 1, False: 0})

print(f'df_features shape: {df_features.shape}')
print(f'df_patients shape: {df_patients.shape}')
print(f'df_questionnaire_items shape: {df_questionnaire_items.shape}')
print(df_features.head())
print(df_patients.head())
print(df_questionnaire_items.head())

df_features shape: (10318, 33)
df_patients shape: (469, 14)
df_questionnaire_items shape: (14070, 5)
   subject_id condition  record_name device_location  \
0           1   Healthy      Relaxed       LeftWrist   
1           1   Healthy      Relaxed      RightWrist   
2           1   Healthy  RelaxedTask       LeftWrist   
3           1   Healthy  RelaxedTask      RightWrist   
4           1   Healthy  StretchHold       LeftWrist   

                                   file_name  rows_expected  n_rows_raw  \
0       timeseries/001_Relaxed_LeftWrist.txt           2048        2048   
1      timeseries/001_Relaxed_RightWrist.txt           2048        2048   
2   timeseries/001_RelaxedTask_LeftWrist.txt           2048        2048   
3  timeseries/001_RelaxedTask_RightWrist.txt           2048        2048   
4   timeseries/001_StretchHold_LeftWrist.txt           1024        1024   

   n_rows_used  row_coverage  sampling_hz  ...  acc_mag_mean  acc_mag_std  \
0         1997           1.0   100

In [4]:
# 3) Train/test split (patient-level, stratified)
X_all = patient_df.drop(columns=['subject_id', 'condition', 'y_pd_binary'], errors='ignore')
X_all = X_all.select_dtypes(include=[np.number])
y_all = patient_df['y_pd_binary']

questionnaire_feature_cols = [c for c in X_all.columns if c.startswith('q_')]
sensor_feature_cols = [c for c in X_all.columns if not c.startswith('q_')]

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all
)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(y_train.value_counts(normalize=True).round(3))
print(f'Total numeric features: {X_all.shape[1]}')
print(f'Sensor/demographic features: {len(sensor_feature_cols)}')
print(f'Questionnaire-derived features: {len(questionnaire_feature_cols)}')

print('\nFirst 5 rows of X_train:')
display(X_train.head())
print('\nFirst 5 rows of X_test:')
display(X_test.head())

X_train: (375, 87) | X_test: (94, 87)
y_pd_binary
1    0.589
0    0.411
Name: proportion, dtype: float64
Total numeric features: 87
Sensor/demographic features: 57
Questionnaire-derived features: 30

First 5 rows of X_train:


,n_rows_raw_mean,n_rows_raw_std,n_rows_used_mean,n_rows_used_std,row_coverage_mean,row_coverage_std,sampling_hz_mean,sampling_hz_std,missing_ratio_before_fill_mean,missing_ratio_before_fill_std,...,q_21,q_22,q_23,q_24,q_25,q_26,q_27,q_28,q_29,q_30
148,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.698344,0.366270,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
287,1303.272727,466.782521,1252.863636,466.726115,1.0,0.0,99.726193,0.384581,0.0,0.0,...,0,1,1,0,0,1,0,0,1,0
294,1303.272727,466.782521,1252.818182,466.754308,1.0,0.0,99.693834,0.360631,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
208,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.689732,0.370770,0.0,0.0,...,0,0,0,0,0,1,0,1,0,0
75,1303.272727,466.782521,1252.818182,466.754308,1.0,0.0,99.699226,0.365302,0.0,0.0,...,0,0,1,0,1,0,0,1,0,0


X_train: (375, 87) | X_test: (94, 87)
y_pd_binary
1    0.589
0    0.411
Name: proportion, dtype: float64
Total numeric features: 87
Sensor/demographic features: 57
Questionnaire-derived features: 30

First 5 rows of X_train:


,n_rows_raw_mean,n_rows_raw_std,n_rows_used_mean,n_rows_used_std,row_coverage_mean,row_coverage_std,sampling_hz_mean,sampling_hz_std,missing_ratio_before_fill_mean,missing_ratio_before_fill_std,...,q_21,q_22,q_23,q_24,q_25,q_26,q_27,q_28,q_29,q_30
148,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.698344,0.366270,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
287,1303.272727,466.782521,1252.863636,466.726115,1.0,0.0,99.726193,0.384581,0.0,0.0,...,0,1,1,0,0,1,0,0,1,0
294,1303.272727,466.782521,1252.818182,466.754308,1.0,0.0,99.693834,0.360631,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
208,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.689732,0.370770,0.0,0.0,...,0,0,0,0,0,1,0,1,0,0
75,1303.272727,466.782521,1252.818182,466.754308,1.0,0.0,99.699226,0.365302,0.0,0.0,...,0,0,1,0,1,0,0,1,0,0



First 5 rows of X_test:


,n_rows_raw_mean,n_rows_raw_std,n_rows_used_mean,n_rows_used_std,row_coverage_mean,row_coverage_std,sampling_hz_mean,sampling_hz_std,missing_ratio_before_fill_mean,missing_ratio_before_fill_std,...,q_21,q_22,q_23,q_24,q_25,q_26,q_27,q_28,q_29,q_30
107,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,100.144191,0.622356,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
416,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.701654,0.362093,0.0,0.0,...,1,1,0,0,1,1,0,0,0,0
258,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.684202,0.369181,0.0,0.0,...,0,0,1,1,0,0,1,0,1,0
452,1303.272727,466.782521,1252.454545,466.564315,1.0,0.0,100.146833,0.625202,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
46,1303.272727,466.782521,1252.772727,466.782802,1.0,0.0,99.702725,0.365012,0.0,0.0,...,0,0,0,1,1,0,1,0,0,0


In [6]:
# 5) Save A/B/C selected datasets for downstream modeling
final_dir = Path('../..') / 'data' / 'final'
final_dir.mkdir(parents=True, exist_ok=True)

for setup_name, payload in setup_results.items():
    setup_slug = setup_name.lower()

    payload['X_train_selected'].to_csv(final_dir / f'X_train_{setup_slug}_selected.csv', index=False)
    payload['X_test_selected'].to_csv(final_dir / f'X_test_{setup_slug}_selected.csv', index=False)

    payload['feature_selection_table'].to_csv(
        final_dir / f'feature_selection_table_{setup_slug}.csv', index=False
    )
    pd.Series(payload['final_feature_list'], name='feature').to_csv(
        final_dir / f'final_feature_list_{setup_slug}.csv', index=False
    )

setup_summary_df.to_csv(final_dir / 'setup_summary_abc.csv', index=False)

print(f"Saved A/B/C selected artifacts in: {final_dir.resolve()}")
print('Generated files:')
for setup_name in sorted(setup_results.keys()):
    setup_slug = setup_name.lower()
    print(f"- X_train_{setup_slug}_selected.csv")
    print(f"- X_test_{setup_slug}_selected.csv")
    print(f"- feature_selection_table_{setup_slug}.csv")
    print(f"- final_feature_list_{setup_slug}.csv")
print('- setup_summary_abc.csv')

Saved A/B/C selected artifacts in: C:\Users\paolo\Desktop\Behavioral-Health-Informatic-Project\data\final
Generated files:
- X_train_a_sensor_demo_selected.csv
- X_test_a_sensor_demo_selected.csv
- feature_selection_table_a_sensor_demo.csv
- final_feature_list_a_sensor_demo.csv
- X_train_b_questionnaire_demo_selected.csv
- X_test_b_questionnaire_demo_selected.csv
- feature_selection_table_b_questionnaire_demo.csv
- final_feature_list_b_questionnaire_demo.csv
- X_train_c_all_modalities_selected.csv
- X_test_c_all_modalities_selected.csv
- feature_selection_table_c_all_modalities.csv
- final_feature_list_c_all_modalities.csv
- setup_summary_abc.csv
